# 030 — Task Selection for Single-Task Distribution Study

Ranks all GPQA tasks in `results/mas/final_dataset/W2_fc` on criteria relevant to the
proposed single-task, high-R distribution study:

- **mean_acc**: task accuracy (mean `correct` over R=30 reps) — want ~0.25–0.65
- **std_acc**: run-to-run accuracy std — want high (contested outcome)
- **mean_h0**: mean initial-vote entropy at round 0 — want high (diverse starts)
- **mean_rounds**: mean rounds-to-consensus — want mid-range
- **ceil_frac**: fraction of reps hitting T=15 — want low
- **conv_rate**: fraction of reps reaching unanimity — want high (clean outcomes)

Final ranking combines these into a composite **selection score** so the best
candidates for the 500–1000 rep experiment float to the top.


In [ ]:
import sys
sys.path.insert(0, '..')

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import entropy as scipy_entropy

BASE = Path('..') / 'results' / 'mas' / 'final_dataset' / 'W2_fc'
T_CEIL = 15

pd.set_option('display.width', 200)
pd.set_option('display.max_columns', 30)
np.set_printoptions(suppress=True)
print('setup ok')

In [ ]:
def initial_entropy(rep):
    options = list(rep['options'].keys())
    votes = [ag['vote'] for ag in rep['trajectory'][0]['phase_b']]
    counts = np.array([votes.count(o) for o in options], dtype=float)
    if counts.sum() == 0:
        return 0.0
    p = counts / counts.sum()
    return float(scipy_entropy(p, base=2))

def load_tasks(base):
    rows = []
    for f in sorted(base.glob('*gpqa_*.json')):
        d = json.loads(f.read_text())
        reps = d['repetitions']
        correct = np.array([r['correct'] for r in reps], dtype=float)
        rounds = np.array([len(r['trajectory']) - 1 for r in reps], dtype=float)
        h0 = np.array([initial_entropy(r) for r in reps])
        converged = np.array([
            all(ag['vote'] == reps[i]['trajectory'][-1]['phase_b'][0]['vote']
                for ag in reps[i]['trajectory'][-1]['phase_b'])
            for i in range(len(reps))
        ], dtype=float)
        rows.append({
            'qid': d['question_id'],
            'n_reps': len(reps),
            'ground_truth': d['ground_truth'],
            'mean_acc': float(correct.mean()),
            'std_acc': float(correct.std()),
            'mean_h0': float(h0.mean()),
            'std_h0': float(h0.std()),
            'mean_rounds': float(rounds.mean()),
            'std_rounds': float(rounds.std()),
            'ceil_frac': float(np.mean(rounds >= T_CEIL)),
            'conv_rate': float(converged.mean()),
        })
    return pd.DataFrame(rows)

tasks = load_tasks(BASE)
print(f'loaded {len(tasks)} GPQA tasks from W2_fc')
print()
print(tasks[['qid','n_reps','mean_acc','std_acc','mean_h0','mean_rounds','ceil_frac','conv_rate']].to_string(index=False))

## Composite selection score

We want tasks that are:
- **contested**: accuracy near 0.5 (penalise distance from 0.45), high std_acc
- **dynamically rich**: high initial entropy, mid-range rounds, low ceiling-hitting
- **clean outcomes**: high convergence rate

Score = normalised sum of six sub-scores, each in [0, 1] after min-max scaling.
Sub-scores:
1. `s_acc_mid`: 1 − 2·|mean_acc − 0.45| / max_range (peaks at mean_acc=0.45)
2. `s_std_acc`: normalised std_acc
3. `s_h0`: normalised mean_h0
4. `s_rounds_mid`: 1 − |mean_rounds − T_MID| / range (peaks at T_MID=7)
5. `s_no_ceil`: 1 − ceil_frac
6. `s_conv`: normalised conv_rate

All sub-scores equal weight (1/6).


In [ ]:
def minmax(x):
    lo, hi = x.min(), x.max()
    if hi == lo:
        return np.zeros_like(x, dtype=float)
    return (x - lo) / (hi - lo)

T_MID = 7.0

df = tasks.copy()
df['s_acc_mid'] = 1 - 2 * np.abs(df['mean_acc'] - 0.45)
df['s_acc_mid'] = minmax(df['s_acc_mid'])
df['s_std_acc'] = minmax(df['std_acc'])
df['s_h0'] = minmax(df['mean_h0'])
rounds_range = df['mean_rounds'].max() - df['mean_rounds'].min()
df['s_rounds_mid'] = 1 - np.abs(df['mean_rounds'] - T_MID) / (rounds_range if rounds_range > 0 else 1)
df['s_rounds_mid'] = minmax(df['s_rounds_mid'])
df['s_no_ceil'] = minmax(1 - df['ceil_frac'])
df['s_conv'] = minmax(df['conv_rate'])

score_cols = ['s_acc_mid', 's_std_acc', 's_h0', 's_rounds_mid', 's_no_ceil', 's_conv']
df['selection_score'] = df[score_cols].mean(axis=1)

df_ranked = df.sort_values('selection_score', ascending=False).reset_index(drop=True)
df_ranked.index += 1

print('Top 15 tasks by composite selection score:')
display_cols = ['qid','mean_acc','std_acc','mean_h0','mean_rounds','ceil_frac','conv_rate','selection_score']
print(df_ranked[display_cols].head(15).to_string())

In [ ]:
print('Full ranking:')
print(df_ranked[display_cols].to_string())

## Criterion breakdown for top 10

Per-criterion sub-scores for the top 10 candidates — helps diagnose why a task ranks high
and whether any criterion dominates.


In [ ]:
top10 = df_ranked.head(10)[['qid'] + score_cols + ['selection_score']]
fig, ax = plt.subplots(figsize=(12, 5))
x = np.arange(len(top10))
width = 0.12
colors = ['#4878CF', '#6ACC65', '#D65F5F', '#B47CC7', '#C4AD66', '#77BEDB']
labels = ['acc_mid', 'std_acc', 'h0', 'rounds_mid', 'no_ceil', 'conv']
for i, (col, label, color) in enumerate(zip(score_cols, labels, colors)):
    ax.bar(x + i * width, top10[col].values, width, label=label, color=color, alpha=0.85)
ax.set_xticks(x + 2.5 * width)
ax.set_xticklabels([f'q{q}' for q in top10['qid']], rotation=45, ha='right')
ax.set_ylabel('Sub-score (0–1)')
ax.set_title('Criterion breakdown — top 10 GPQA tasks (W2 fc)')
ax.legend(loc='upper right', fontsize=8)
ax.axhline(1/6, color='k', lw=0.8, ls='--', label='equal-weight floor')
plt.tight_layout()
plt.show()

## Distribution overview — top 10

Empirical distribution of `correct` over R=30 reps for each top-10 task.
This is exactly the distribution the high-R experiment will resolve in detail.


In [ ]:
top10_qids = df_ranked.head(10)['qid'].tolist()
fig, axes = plt.subplots(2, 5, figsize=(16, 6))
axes = axes.flatten()

for ax, qid in zip(axes, top10_qids):
    f = next(BASE.glob(f'*gpqa_*_q{qid}_*.json'), None)
    if f is None:
        ax.set_title(f'q{qid} not found'); continue
    d = json.loads(f.read_text())
    reps = d['repetitions']
    acc = float(np.mean([r['correct'] for r in reps]))
    h0_vals = [initial_entropy(r) for r in reps]
    rounds_vals = [len(r['trajectory']) - 1 for r in reps]
    correct_vals = [r['correct'] for r in reps]
    ax.bar(['wrong', 'correct'], [sum(not c for c in correct_vals), sum(correct_vals)],
           color=['#D65F5F', '#6ACC65'])
    ax.set_title(f'q{qid}  acc={acc:.2f}  h0={np.mean(h0_vals):.2f}', fontsize=9)
    ax.set_ylim(0, len(reps))

plt.suptitle('Outcome distribution over R=30 reps — top 10 tasks (W2 fc)', fontsize=11)
plt.tight_layout()
plt.show()

## Scatter: accuracy vs initial entropy

Placing all 35 tasks in the (mean_h0, mean_acc) plane — the axes of the proposed
distribution experiment. Top-5 candidates highlighted.


In [ ]:
top5_qids = set(df_ranked.head(5)['qid'].tolist())

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(df['mean_h0'], df['mean_acc'],
                c=df['selection_score'], cmap='viridis', s=80, zorder=3)
plt.colorbar(sc, ax=ax, label='selection score')

for _, row in df.iterrows():
    style = dict(fontsize=8, ha='left', va='bottom')
    if str(row['qid']) in [str(q) for q in top5_qids]:
        ax.annotate(f"q{row['qid']}", (row['mean_h0'], row['mean_acc']),
                    xytext=(4, 4), textcoords='offset points',
                    fontsize=8, fontweight='bold', color='#D65F5F')
    else:
        ax.annotate(f"q{row['qid']}", (row['mean_h0'], row['mean_acc']),
                    xytext=(4, 4), textcoords='offset points', **style)

ax.axhspan(0.25, 0.65, alpha=0.08, color='green', label='target acc band [0.25, 0.65]')
ax.set_xlabel('Mean initial entropy H₀ (bits)')
ax.set_ylabel('Mean accuracy')
ax.set_title('GPQA tasks (W2 fc) — accuracy vs initial diversity\nbold/red = top-5 candidates')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## Recommendation

The table and plots above give enough information to select the single task for the
high-R experiment. Key considerations for the final choice:

- Prefer a task in the **green band** (0.25–0.65 accuracy) with **high H₀** and **high conv_rate**
- Avoid tasks with `ceil_frac > 0.3` — high ceiling-hitting means many runs are truncated
- The composite score balances all these; top-3 are the natural first candidates to examine

After selecting a task, the run command is:
```
python run_mas.py --dataset gpqa --index <qid> --model mistral-medium \
    --n 4 --t 15 --w 2 --topology fc --r 1000 --workers 8 \
    --early-stopping --u 3 --run-name single_task_dist_<qid>
```
